<a href="https://colab.research.google.com/github/Innovatewithapple/Pipelines-CrossValidation-Regularisation/blob/main/ClassificationPipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# basic
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# model + preprocessing
from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

#model
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [10]:
df = pd.read_csv('/content/student_mental_health_burnout.csv')
df.sample(10)

,student_id,age,gender,course,year,daily_study_hours,daily_sleep_hours,screen_time_hours,stress_level,anxiety_score,depression_score,academic_pressure_score,financial_stress_score,social_support_score,physical_activity_hours,sleep_quality,attendance_percentage,cgpa,internet_quality,burnout_level
63592,163593,18,Male,MBA,4th,9.9,5.4,8.6,Low,4,1,1,1,8,0.8,Good,95.9,9.99,Average,Low
27566,127567,25,Male,BBA,3rd,9.7,4.4,3.5,Medium,10,4,7,5,9,1.4,Average,63.2,4.97,Poor,Low
13345,113346,19,Female,MBA,4th,1.8,5.0,4.4,Medium,9,10,2,4,2,2.0,Good,78.6,4.75,Good,Low
113408,213409,19,Female,MBA,1st,3.3,5.9,10.9,High,2,5,4,4,4,1.6,Good,54.7,9.50,Average,Medium
32682,132683,25,Female,BSc,3rd,4.9,8.8,11.2,Medium,10,2,9,6,8,1.9,Average,73.8,9.33,Poor,Medium
23143,123144,23,Female,MCA,1st,1.4,7.5,2.1,High,8,3,10,2,8,1.8,Poor,53.3,4.74,Average,High
68500,168501,20,Female,BTech,4th,8.1,4.2,6.9,Medium,6,5,5,8,2,0.3,Good,74.8,4.56,Poor,Low
102614,202615,21,Male,BTech,2nd,2.4,7.8,1.3,Low,4,1,8,1,8,1.8,Good,97.7,5.70,Good,High
85582,185583,23,Other,BCA,4th,3.6,7.2,11.6,Medium,4,5,6,7,3,1.1,Average,90.1,6.02,Poor,High
86379,186380,21,Other,BSc,4th,8.0,6.9,7.5,Low,8,2,9,9,8,1.1,Poor,52.2,6.76,Average,Low


In [4]:
df.describe()

,student_id,age,daily_study_hours,daily_sleep_hours,screen_time_hours,anxiety_score,depression_score,academic_pressure_score,financial_stress_score,social_support_score,physical_activity_hours,attendance_percentage,cgpa
count,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000
mean,175000.500000,21.000380,5.507869,6.499361,6.502819,5.493907,5.497360,5.507427,5.496027,5.516060,0.998115,75.009528,6.997389
std,43301.414527,2.581216,2.595592,1.443859,3.178948,2.872607,2.869022,2.875524,2.864698,2.870493,0.578866,14.409510,1.732180
min,100001.000000,17.000000,1.000000,4.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,50.000000,4.000000
25%,137500.750000,19.000000,3.300000,5.200000,3.700000,3.000000,3.000000,3.000000,3.000000,3.000000,0.500000,62.500000,5.500000
50%,175000.500000,21.000000,5.500000,6.500000,6.500000,5.000000,5.000000,6.000000,6.000000,6.000000,1.000000,75.000000,6.990000
75%,212500.250000,23.000000,7.700000,7.700000,9.300000,8.000000,8.000000,8.000000,8.000000,8.000000,1.500000,87.500000,8.500000
max,250000.000000,25.000000,10.000000,9.000000,12.000000,10.000000,10.000000,10.000000,10.000000,10.000000,2.000000,100.000000,10.000000


In [11]:
df.sample(10)

,student_id,age,gender,course,year,daily_study_hours,daily_sleep_hours,screen_time_hours,stress_level,anxiety_score,depression_score,academic_pressure_score,financial_stress_score,social_support_score,physical_activity_hours,sleep_quality,attendance_percentage,cgpa,internet_quality,burnout_level
144438,244439,18,Other,BTech,1st,2.3,8.2,6.8,Medium,8,8,2,3,4,0.4,Good,81.5,8.66,Good,Low
99686,199687,23,Male,MCA,4th,8.6,6.5,2.8,Low,8,1,6,6,3,0.9,Good,92.1,5.16,Poor,High
49789,149790,17,Female,BBA,2nd,5.8,8.5,10.2,Low,1,3,9,8,7,1.5,Poor,55.1,7.03,Good,Medium
7867,107868,22,Female,BTech,3rd,9.5,5.0,11.8,Low,4,8,1,10,8,0.4,Average,96.9,6.35,Average,Low
20725,120726,17,Other,BTech,3rd,3.2,5.0,3.4,High,10,9,7,4,10,0.8,Poor,79.5,4.65,Good,High
47927,147928,21,Male,MBA,1st,9.4,5.5,6.6,Medium,1,9,6,4,9,1.4,Average,62.7,9.54,Average,Low
78754,178755,18,Male,MBA,2nd,9.1,5.1,4.3,Low,10,3,10,7,4,0.9,Poor,96.5,7.79,Average,Low
12143,112144,20,Other,BCA,4th,9.4,8.1,2.9,High,2,7,8,9,1,0.3,Good,55.2,9.28,Poor,Low
100534,200535,20,Other,MBA,1st,5.0,7.8,10.4,Medium,2,6,4,9,7,0.4,Good,71.3,9.97,Average,Low
113715,213716,23,Female,MBA,3rd,7.6,6.8,4.4,Low,8,6,8,9,5,0.9,Good,70.6,5.94,Average,Medium


In [12]:
x = df.drop(columns=['student_id','course'])
y = df['burnout_level']

In [14]:
labelencode = LabelEncoder()
y = labelencode.fit_transform(y)

In [15]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=42)

In [20]:
#transorm
scalingColumns = ['age','daily_study_hours','daily_sleep_hours','screen_time_hours','anxiety_score','depression_score','academic_pressure_score','financial_stress_score','social_support_score','physical_activity_hours','attendance_percentage','cgpa']
encodeColumns = ['gender','year','stress_level','sleep_quality','internet_quality']
preprocessor = ColumnTransformer([
    ('nums',StandardScaler(),scalingColumns),
    ('cat',OneHotEncoder(),encodeColumns)
])

In [21]:
#Logistic
pipeline = Pipeline([
    ('preprocess',preprocessor),
    ('model',LogisticRegression())
])

In [22]:
pipeline.fit(x_train,y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('nums', StandardScaler(),
                                                  ['age', 'daily_study_hours',
                                                   'daily_sleep_hours',
                                                   'screen_time_hours',
                                                   'anxiety_score',
                                                   'depression_score',
                                                   'academic_pressure_score',
                                                   'financial_stress_score',
                                                   'social_support_score',
                                                   'physical_activity_hours',
                                                   'attendance_percentage',
                                                   'cgpa']),
                                                 ('cat', OneHotEncoder(),
                                                  ['gender', 'year',
                                                   'stress_level',
                                                   'sleep_quality',
                                                   'internet_quality'])])),
                ('model', LogisticRegression())])

In [23]:
cross_score = cross_val_score(pipeline,x_train,y_train,cv=5,scoring='f1_weighted')
print(cross_score)

[0.32988589 0.32629949 0.32787962 0.33148229 0.32959371]


In [24]:
cross_score = cross_val_score(pipeline,x_train,y_train,cv=5,scoring='accuracy')
print(cross_score)

[0.33547619 0.331      0.33166667 0.33561905 0.33304762]


In [27]:
# SVM
pipeline_svm = Pipeline([
    ('preprocess',preprocessor),
    ('model',SVC())
])

In [ ]:
pipeline_svm.fit(x_train,y_train)